# Unit 04 — Worldle Notebook

This notebook builds a Worldle-like game using local `Resources/wdo` helpers.

- Mystery country polygon is shown on a map.
- You guess countries from a searchable widget.
- Wrong guesses show name, direction, and distance.


In [ ]:
from pathlib import Path
import sys
import json
import random
import base64

# Make local package importable when notebook runs inside Assignments/04-Worldle
project_root = Path.cwd()
resources_path = project_root / "Resources"
if str(resources_path) not in sys.path:
    sys.path.insert(0, str(resources_path))

from wdo.maps.leaflet_helpers import make_map, add_geojson, fit_map_to_geojson
from wdo.games.worldle import choose_target, guess_feedback, format_feedback


In [ ]:
data_path = Path("Resources/Data/countries_export.json")
if not data_path.exists():
    print("Missing file: Resources/Data/countries_export.json")
    print("Put the countries GeoJSON there before playing the full game.")
    countries = {"type": "FeatureCollection", "features": []}
else:
    with data_path.open("r", encoding="utf-8") as f:
        countries = json.load(f)

features = countries.get("features", [])
print(f"Loaded {len(features)} features")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

if not features:
    display(HTML("<b>Data file missing, so game UI is disabled.</b>"))
else:
    # Build lookup by country name
    by_name = {f.get('properties', {}).get('ADMIN', 'Unknown'): f for f in features}
    country_names = sorted(by_name.keys())

    target = choose_target(features, seed=None)

    m = make_map(center=(20, 0), zoom=2)
    target_layer_data = {"type": "FeatureCollection", "features": [target]}
    add_geojson(m, target_layer_data, name="Mystery Country")
    fit_map_to_geojson(m, target_layer_data)

    # Hide identifying labels by not adding popups/tooltips
    chooser = widgets.Combobox(options=country_names, placeholder="Type country name", description="Guess:")
    guess_button = widgets.Button(description="Guess", button_style="primary")
    output = widgets.Output()

    history_rows = []

    def render_history():
        rows = "".join(history_rows)
        return f"<table style='border-collapse:collapse; width:100%;'>"                f"<tr><th style='text-align:left'>Country</th><th>Direction</th><th>Distance</th></tr>{rows}</table>"

    def on_guess(_):
        name = chooser.value
        if name not in by_name:
            with output:
                print("Pick a valid country from the list.")
            return

        guess = by_name[name]
        fb = guess_feedback(guess, target)

        if fb["correct"]:
            history_rows.append(f"<tr><td>{name}</td><td>✅</td><td>0 km</td></tr>")
            with output:
                output.clear_output()
                display(HTML(render_history()))
                display(HTML("<h3 style='color:green'>You got it! 🎉</h3>"))
            guess_button.disabled = True
            chooser.disabled = True
            return

        history_rows.append(
            f"<tr><td>{name}</td><td>{fb['arrow']} {fb['compass']}</td><td>{fb['distance_km']:.1f} km</td></tr>"
        )
        with output:
            output.clear_output()
            display(HTML(render_history()))
            print(format_feedback(fb))

    guess_button.on_click(on_guess)

    display(m)
    display(widgets.HBox([chooser, guess_button]))
    display(output)


## Write-up (Fill before submission)

### Functions implemented
- bbox_from_feature
- haversine_distance_km / initial_bearing / bearing_to_compass
- make_map / add_geojson / fit_map_to_geojson
- choose_target / feature_center / guess_feedback / format_feedback

### Alias / ISO bridge notes
- Add notes on ISO-3 (polygons) to ISO-2 (flags) bridging and any aliases you used.

### Known bugs
- List edge cases or limitations found during testing.

### Screenshot placeholder
- Insert one screenshot of a completed game round.

### Polish features
- List at least two polish features you added.
